# Notebook for manual implementation of Batch Norm

### Computational Graph
#### $ f_{1} = E[z_{i}] $
#### $ f_{2i} = z_{i} - f_{1} $

In [69]:
import numpy as np
import os
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import matplotlib.pyplot as plt
import mnist1d
import random

In [70]:
def compute_f_1(z_i):
    result = np.mean(z_i)
    return result

In [71]:
def test_compute_f_1():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_1(input)
    assert(result == 3)

test_compute_f_1()

In [72]:
def compute_f_2i(z_i, f_1):
    result = z_i - f_1
    return result

In [73]:
def test_compute_f_2i():
    input = np.array([ 2, 2, 2, 4, 4, 4 ])
    result = compute_f_2i(input, compute_f_1(input))
    expected = np.array([2-3, 2-3, 2-3, 4-3, 4-3, 4-3])
    comparison = result == expected
    assert(comparison.all())

test_compute_f_2i()

### Computational Graph

#### We have the $ z_{i} $ now with the mean normalised; see `test_compute_f_2i()`

#### $ f_{3i} = f_{2i}^{2} $
#### $ f_{4} = E(f_{3i}) $
#### $ f_{5} = \sqrt{f_{4} + \epsilon} $
#### $ f_{6} = 1 / f_{5} $

In [74]:
def compute_f_3(f_2):
    result = np.square(f_2)
    return result

def compute_f_4(f_3i):
    result = np.mean(f_3i)
    return result

def compute_f_5(f_4):
    assert(f_4 >= 0)
    epsilon = 1e-5
    result = np.sqrt(f_4 + epsilon)
    return result

def compute_f_6(f_5):
    result = 1 / f_5
    return result

In [75]:
def test_compute_f3():
    input = np.array([2, 0, -1])
    result = compute_f_3(input)
    expected = np.array([4, 0, 1])
    comparison = result == expected
    assert(comparison.all())

test_compute_f3()

In [76]:
def test_compute_f_4():
    input = np.array([4, 0, 1])
    result = compute_f_4(input)
    assert((4+0+1)/3 == result)

test_compute_f_4()

In [77]:
def test_compute_f_5():
    assert(compute_f_5(0) > 0)
    result = compute_f_5(2)
    print(result)
    assert(1.41 < result)
    assert(result < 1.42)

test_compute_f_5()

1.4142170979025817


In [82]:
def test_compute_f_6():
    input = 0
    result1 = compute_f_5(input)
    assert(result1 > 0.003)
    result2 = compute_f_6(result1)
    assert(result2 > 316)
    assert(result2 < 317)

test_compute_f_6()
    

### Computational Graph

#### $ f_{7i} = f_{2i} * f_{6} $
#### $ z_{i}' = f_{7i} \times \gamma + \delta $